# RQ2: Topics and Narratives

**Research Question 2 - What distinct topics and narratives emerge on each platform?**

This module identifies the dominant themes and framings in the discourse using:

- **Word-frequency analysis** (overall and by stance)
- **LDA** (Latent Dirichlet Allocation) and **NMF** (Non-negative Matrix Factorization) topic models
- **Word clouds** by stance and by sentiment
- **TF-IDF** distinctive-term extraction per stance

It loads the sentiment-enriched data produced by module 02 (falling back to
recomputing VADER from `data/` if those files are absent).

**Stance labels:** `P` = Pro-Palestine, `I` = Pro-Israel, `N` = Neutral.

## 1. Setup, Paths and Data Loader

In [ ]:
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from wordcloud import WordCloud, STOPWORDS
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 8)

STANCE_ORDER = ["P", "I", "N"]
STANCE_NAMES = {"P": "Pro-Palestine", "I": "Pro-Israel", "N": "Neutral"}


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "reddit_labeled.csv").exists():
            return candidate
    return here


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
SENTIMENT_DIR = REPO_ROOT / "02_emotional_tone_analysis" / "outputs"
OUTPUT_DIR = REPO_ROOT / "03_topics_and_narratives" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _vader_labels(texts):
    """Compute (vader_label, vader_compound) for a text Series (fallback path)."""
    an = SentimentIntensityAnalyzer()

    def lab(t):
        if not isinstance(t, str) or t == "":
            return ("neutral", 0.0)
        c = an.polarity_scores(t)["compound"]
        label = "positive" if c >= 0.05 else "negative" if c <= -0.05 else "neutral"
        return (label, c)

    res = texts.fillna("").astype(str).map(lab)
    return pd.DataFrame(res.tolist(), columns=["vader_label", "vader_compound"],
                        index=texts.index)


def load_sentiment_data():
    """Load module-02 sentiment outputs if present, else recompute VADER."""
    rp = SENTIMENT_DIR / "reddit_with_sentiment.csv"
    yp = SENTIMENT_DIR / "youtube_with_sentiment.csv"
    if rp.exists() and yp.exists():
        print("Loaded sentiment-enriched data from module 02 outputs.")
        reddit, youtube = pd.read_csv(rp), pd.read_csv(yp)
    else:
        print("Module 02 outputs not found - loading data/ and computing VADER (fallback)...")
        reddit = pd.read_csv(DATA_DIR / "reddit_labeled.csv")
        youtube = pd.read_csv(DATA_DIR / "youtube_labeled.csv")
        reddit = reddit[reddit["Label"].isin(STANCE_ORDER)].copy()
        youtube = youtube[youtube["Label"].isin(STANCE_ORDER)].copy()
        reddit = pd.concat([reddit, _vader_labels(reddit["self_text"])], axis=1)
        youtube = pd.concat([youtube, _vader_labels(youtube["text"])], axis=1)

    reddit = reddit[reddit["Label"].isin(STANCE_ORDER)].copy()
    youtube = youtube[youtube["Label"].isin(STANCE_ORDER)].copy()
    for col in ["score", "post_score", "post_upvote_ratio", "controversiality"]:
        if col in reddit.columns:
            reddit[col] = pd.to_numeric(reddit[col], errors="coerce")
    if "likeCount" in youtube.columns:
        youtube["likeCount"] = pd.to_numeric(youtube["likeCount"], errors="coerce")
    return reddit, youtube


print(f"Output dir: {OUTPUT_DIR}")

## 2. Load and Preprocess Text

In [ ]:
reddit_df, youtube_df = load_sentiment_data()
print(f"Reddit : {len(reddit_df):,} rows")
print(f"YouTube: {len(youtube_df):,} rows")

REDDIT_TEXT, YOUTUBE_TEXT = "self_text", "text"


def clean_text(text):
    """Lowercase, strip URLs/mentions/punctuation for frequency & topic models."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\@\w+|\#", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


reddit_df["cleaned_text"] = reddit_df[REDDIT_TEXT].map(clean_text)
youtube_df["cleaned_text"] = youtube_df[YOUTUBE_TEXT].map(clean_text)

# Keep documents with enough content for meaningful topic modelling
reddit_topics_df = reddit_df[reddit_df["cleaned_text"].str.len() > 20]
youtube_topics_df = youtube_df[youtube_df["cleaned_text"].str.len() > 20]
print(f"Reddit docs for topic modelling : {len(reddit_topics_df):,}")
print(f"YouTube docs for topic modelling: {len(youtube_topics_df):,}")

## 3. Word Frequency Analysis

Domain terms (israel, gaza, hamas, ...) are added to the stopword list so the
*differentiating* vocabulary surfaces instead of the obvious shared terms.

In [ ]:
DOMAIN_STOPWORDS = [
    "israel", "israeli", "palestine", "palestinian", "hamas", "gaza", "war",
    "conflict", "just", "like", "people", "know", "think", "going", "said",
    "really", "also", "would", "could", "one", "two", "even", "make", "get",
    "want", "see", "say", "tell", "much", "many", "thing", "way", "time",
]
ALL_STOPWORDS = sorted(set(CountVectorizer(stop_words="english").get_stop_words())
                       | set(DOMAIN_STOPWORDS))


def get_top_words(texts, n=20):
    vec = CountVectorizer(max_features=n, stop_words=ALL_STOPWORDS)
    X = vec.fit_transform(texts)
    freq = dict(zip(vec.get_feature_names_out(), np.asarray(X.sum(axis=0)).ravel()))
    return Counter(freq).most_common(n)


reddit_top_words = get_top_words(reddit_topics_df["cleaned_text"], 20)
youtube_top_words = get_top_words(youtube_topics_df["cleaned_text"], 20)

print("TOP 20 WORDS - REDDIT:")
for w, c in reddit_top_words:
    print(f"  {w}: {c:,}")
print("\nTOP 20 WORDS - YOUTUBE:")
for w, c in youtube_top_words:
    print(f"  {w}: {c:,}")

print("\nTOP WORDS BY STANCE - REDDIT:")
for stance in STANCE_ORDER:
    texts = reddit_topics_df[reddit_topics_df["Label"] == stance]["cleaned_text"]
    if len(texts):
        words = ", ".join(w for w, _ in get_top_words(texts, 10))
        print(f"  {STANCE_NAMES[stance]}: {words}")

## 4. Topic Modeling - LDA

In [ ]:
N_TOPICS = 5
N_TOP_WORDS = 10


def fit_lda(texts):
    vec = CountVectorizer(max_df=0.95, min_df=5, max_features=1000, stop_words="english")
    tf = vec.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42,
                                    max_iter=20, learning_method="online",
                                    batch_size=4096)
    lda.fit(tf)
    return lda, vec


def show_topics(model, vec, title):
    print(f"\n{title} - top {N_TOP_WORDS} words per topic:")
    names = vec.get_feature_names_out()
    for idx, topic in enumerate(model.components_):
        words = [names[i] for i in topic.argsort()[-N_TOP_WORDS:][::-1]]
        print(f"  Topic {idx + 1}: {', '.join(words)}")


print("Fitting LDA (Reddit)...")
reddit_lda, reddit_vec = fit_lda(reddit_topics_df["cleaned_text"])
show_topics(reddit_lda, reddit_vec, "REDDIT LDA")

print("\nFitting LDA (YouTube)...")
youtube_lda, youtube_vec = fit_lda(youtube_topics_df["cleaned_text"])
show_topics(youtube_lda, youtube_vec, "YOUTUBE LDA")

## 5. Topic Modeling - NMF

In [ ]:
def fit_nmf(texts):
    vec = TfidfVectorizer(max_df=0.95, min_df=5, max_features=1000, stop_words="english")
    tfidf = vec.fit_transform(texts)
    nmf = NMF(n_components=N_TOPICS, random_state=42, max_iter=300, init="nndsvda")
    nmf.fit(tfidf)
    return nmf, vec


print("Fitting NMF (Reddit)...")
reddit_nmf, reddit_tfidf_vec = fit_nmf(reddit_topics_df["cleaned_text"])
show_topics(reddit_nmf, reddit_tfidf_vec, "REDDIT NMF")

print("\nFitting NMF (YouTube)...")
youtube_nmf, youtube_tfidf_vec = fit_nmf(youtube_topics_df["cleaned_text"])
show_topics(youtube_nmf, youtube_tfidf_vec, "YOUTUBE NMF")

## 6. Topic Visualizations

In [ ]:
# Word frequency bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, top_words, title, color in [
    (axes[0], reddit_top_words, "Reddit: Top 20 Words", "#3498db"),
    (axes[1], youtube_top_words, "YouTube: Top 20 Words", "#e74c3c"),
]:
    words = [w for w, _ in top_words]
    counts = [c for _, c in top_words]
    ax.barh(words, counts, color=color, alpha=0.85, edgecolor="black")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Frequency")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# LDA topic-word weight heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, lda, vec, title in [
    (axes[0], reddit_lda, reddit_vec, "Reddit: LDA Topics"),
    (axes[1], youtube_lda, youtube_vec, "YouTube: LDA Topics"),
]:
    names = vec.get_feature_names_out()
    top_idx = lda.components_[0].argsort()[-10:][::-1]
    data = [topic[top_idx] for topic in lda.components_]
    sns.heatmap(data, xticklabels=[names[i] for i in top_idx],
                yticklabels=[f"Topic {i+1}" for i in range(N_TOPICS)],
                cmap="YlOrRd", ax=ax, cbar_kws={"label": "Weight"})
    ax.set_title(title, fontsize=12, fontweight="bold")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Word Clouds

Word clouds are qualitative; to keep them fast they are built from a stratified
sample of up to 50,000 comments per group (the full corpus would build the same
picture far more slowly).

In [ ]:
WC_SAMPLE = 50000
custom_stopwords = set(STOPWORDS) | set(DOMAIN_STOPWORDS)


def clean_for_wc(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def corpus_text(df, text_col, mask):
    sub = df[mask]
    if len(sub) > WC_SAMPLE:
        sub = sub.sample(WC_SAMPLE, random_state=42)
    return " ".join(sub[text_col].fillna("").map(clean_for_wc))


def render_wordclouds(df, text_col, groups, title, colormap):
    fig, axes = plt.subplots(1, len(groups), figsize=(6 * len(groups), 6))
    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.02)
    for ax, (mask, label, n) in zip(axes, groups):
        text = corpus_text(df, text_col, mask)
        if text.strip():
            wc = WordCloud(width=800, height=600, background_color="white",
                           stopwords=custom_stopwords, colormap=colormap,
                           max_words=100, relative_scaling=0.5,
                           min_font_size=10).generate(text)
            ax.imshow(wc, interpolation="bilinear")
        else:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=20)
        ax.set_title(f"{label}\n(n={n:,})", fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# By stance
for df, text_col, plat, cmap in [(reddit_df, REDDIT_TEXT, "Reddit", "viridis"),
                                 (youtube_df, YOUTUBE_TEXT, "YouTube", "plasma")]:
    groups = [(df["Label"] == s, STANCE_NAMES[s], (df["Label"] == s).sum())
              for s in STANCE_ORDER]
    render_wordclouds(df, text_col, groups, f"{plat}: Word Clouds by Stance", cmap)

In [ ]:
# By sentiment (vader_label is lowercase: positive / negative / neutral)
SENTIMENTS = [("positive", "Positive"), ("negative", "Negative"), ("neutral", "Neutral")]
for df, text_col, plat in [(reddit_df, REDDIT_TEXT, "Reddit"),
                           (youtube_df, YOUTUBE_TEXT, "YouTube")]:
    groups = [(df["vader_label"] == code, name, (df["vader_label"] == code).sum())
              for code, name in SENTIMENTS]
    render_wordclouds(df, text_col, groups, f"{plat}: Word Clouds by Sentiment (VADER)", "RdYlGn")

In [ ]:
# Platform comparison (overall)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Platform Comparison: Overall Word Clouds", fontsize=16, fontweight="bold")
for ax, df, text_col, plat, cmap in [
    (axes[0], reddit_df, REDDIT_TEXT, "Reddit", "Blues"),
    (axes[1], youtube_df, YOUTUBE_TEXT, "YouTube", "Reds"),
]:
    text = corpus_text(df, text_col, df.index == df.index)  # all rows (sampled)
    wc = WordCloud(width=800, height=800, background_color="white",
                   stopwords=custom_stopwords, colormap=cmap, max_words=150,
                   relative_scaling=0.5, min_font_size=10).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"{plat} (n={len(df):,})", fontsize=14, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. TF-IDF Distinctive Terms by Stance

Highest mean-TF-IDF terms per stance highlight the vocabulary that distinguishes
each side's framing.

In [ ]:
for name, df in [("Reddit", reddit_topics_df), ("YouTube", youtube_topics_df)]:
    print(f"\n{name} - distinctive TF-IDF terms per stance:")
    for stance in STANCE_ORDER:
        texts = df[df["Label"] == stance]["cleaned_text"]
        if len(texts) < 5:
            continue
        vec = TfidfVectorizer(max_features=2000, stop_words=ALL_STOPWORDS)
        tfidf = vec.fit_transform(texts)
        means = np.asarray(tfidf.mean(axis=0)).ravel()
        names = vec.get_feature_names_out()
        top = [names[i] for i in means.argsort()[-8:][::-1]]
        print(f"  {STANCE_NAMES[stance]}: {', '.join(top)}")

## 8b. Topic Coherence (c_v) and Choosing K

Rather than fixing the number of topics arbitrarily, we quantify topic quality with
the **c_v coherence** score (gensim) and compare LDA vs NMF. We also sweep the number
of LDA topics K and pick the most coherent value. Coherence is estimated on a sample
of the cleaned corpus.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

COH_SAMPLE = 20000


def coherence_of(topics_words, texts_tokens):
    d = Dictionary(texts_tokens)
    cm = CoherenceModel(topics=topics_words, texts=texts_tokens, dictionary=d, coherence="c_v")
    return cm.get_coherence()


def top_words_components(model, vec, n=10):
    names = vec.get_feature_names_out()
    return [[names[i] for i in t.argsort()[-n:][::-1]] for t in model.components_]


def sample_tokens(df, n=COH_SAMPLE):
    texts = df["cleaned_text"] if len(df) <= n else df["cleaned_text"].sample(n, random_state=42)
    return [t.split() for t in texts if isinstance(t, str) and t]


for name, tdf, lda, lvec, nmf, nvec in [
    ("Reddit", reddit_topics_df, reddit_lda, reddit_vec, reddit_nmf, reddit_tfidf_vec),
    ("YouTube", youtube_topics_df, youtube_lda, youtube_vec, youtube_nmf, youtube_tfidf_vec),
]:
    toks = sample_tokens(tdf)
    lda_c = coherence_of(top_words_components(lda, lvec), toks)
    nmf_c = coherence_of(top_words_components(nmf, nvec), toks)
    print(f"{name}: c_v coherence  LDA={lda_c:.4f} | NMF={nmf_c:.4f}")

In [ ]:
# Coherence vs number of LDA topics K
Ks = [3, 4, 5, 6, 7, 8]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, tdf) in zip(axes, [("Reddit", reddit_topics_df), ("YouTube", youtube_topics_df)]):
    texts = tdf["cleaned_text"].sample(min(len(tdf), COH_SAMPLE), random_state=42)
    toks = [t.split() for t in texts if isinstance(t, str) and t]
    vec = CountVectorizer(max_df=0.95, min_df=5, max_features=1000, stop_words="english")
    tf = vec.fit_transform(texts)
    names = vec.get_feature_names_out()
    scores = []
    for k in Ks:
        lda = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=20,
                                        learning_method="online", batch_size=4096).fit(tf)
        tw = [[names[i] for i in t.argsort()[-10:][::-1]] for t in lda.components_]
        scores.append(coherence_of(tw, toks))
    best = Ks[int(np.argmax(scores))]
    ax.plot(Ks, scores, marker="o", color="#8e44ad")
    ax.axvline(best, color="red", linestyle="--", alpha=0.6, label=f"best K={best}")
    ax.set_title(f"{name}: LDA coherence (c_v) vs K", fontsize=12, fontweight="bold")
    ax.set_xlabel("Number of topics (K)"); ax.set_ylabel("c_v"); ax.legend()
    print(f"{name}: most coherent K = {best} (c_v={max(scores):.4f})")
plt.tight_layout()
plt.show()

## 8c. BERTopic (embedding-based topics)

A modern alternative to LDA/NMF: sentence-transformer embeddings + UMAP + HDBSCAN
discover topics from semantic structure (and the number of topics is data-driven, not
fixed). Run on a per-platform sample for tractable CPU runtime; the embedding model is
downloaded once on first run.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

BERTOPIC_SAMPLE = 12000
_embedder = SentenceTransformer("all-MiniLM-L6-v2")

for name, tdf, tcol in [("Reddit", reddit_topics_df, REDDIT_TEXT),
                        ("YouTube", youtube_topics_df, YOUTUBE_TEXT)]:
    s = tdf if len(tdf) <= BERTOPIC_SAMPLE else tdf.sample(BERTOPIC_SAMPLE, random_state=42)
    docs = s[tcol].fillna("").astype(str).str.slice(0, 400).tolist()
    emb = _embedder.encode(docs, batch_size=64, show_progress_bar=False)
    tm = BERTopic(embedding_model=_embedder, min_topic_size=50, nr_topics="auto", verbose=False)
    topics, _ = tm.fit_transform(docs, emb)
    n_topics = len([t for t in set(topics) if t != -1])
    print(f"\n{name}: BERTopic found {n_topics} topics (showing up to 7):")
    for tid in tm.get_topic_info()["Topic"].head(8):
        if tid == -1:
            continue
        words = [w for w, _ in tm.get_topic(tid)][:8]
        print(f"  Topic {tid}: {', '.join(words)}")

## 9. Export Topic Data

In [ ]:
reddit_out = OUTPUT_DIR / "reddit_topics_cleaned.csv"
youtube_out = OUTPUT_DIR / "youtube_topics_cleaned.csv"
reddit_topics_df[["cleaned_text", "Label", "vader_label"]].to_csv(reddit_out, index=False)
youtube_topics_df[["cleaned_text", "Label", "vader_label"]].to_csv(youtube_out, index=False)
print(f"Exported: {reddit_out}")
print(f"Exported: {youtube_out}")

## Summary - RQ2

This module surfaces *what* each platform and stance talks about. Interpretation is
left to the freshly computed output above - read the actual top words, LDA/NMF
topics and TF-IDF terms rather than any pre-written conclusion.

- **Frequency & TF-IDF** show the differentiating vocabulary once shared domain
  terms (israel, gaza, hamas, ...) are removed, overall and per stance.
- **LDA and NMF** each recover five themes per platform; comparing the two models
  guards against artefacts of any single algorithm.
- **Word clouds** by stance and by sentiment give a qualitative read on how the
  communities differ (built on a stratified sample - see the note above).

Cleaned topic data is exported for reference and downstream reuse.